In [1]:
import os
import sys

import langchain
import langchain_core
import langchain_openai

from dotenv import load_dotenv

load_dotenv()

# 가상환경 확인
print(f"Python 실행 경로: {sys.executable}")
if ".venv" in sys.executable:
    
    print("[OK] .venv 가상환경 사용 중")
else:
    print("[WARN] .venv 가상환경이 아닌 인터프리터가 선택되어 있습니다")

print("[OK] OpenAI API 키 설정됨" if os.getenv("OPENAI_API_KEY") else "[ERROR] OpenAI API 키 미설정")
print("[OK] Anthropic API 키 설정됨" if os.getenv("ANTHROPIC_API_KEY") else "[ERROR] Anthropic API 키 미설정")
print("[OK] Google API 키 설정됨" if os.getenv("GOOGLE_API_KEY") else "[ERROR] Google API 키 미설정")
print("[OK] LangSmith 키 설정됨" if os.getenv("LANGSMITH_API_KEY") else "[ERROR] LangSmith 키 미설정")

print(f"LangChain: {langchain.__version__}")
print(f"LangChain Core: {langchain_core.__version__}")
print(f"LangChain OpenAI: {langchain_openai.__version__}")

Python 실행 경로: d:\yugyeong\hanwha_0902\days\ex_0918\.venv\Scripts\python.exe
[OK] .venv 가상환경 사용 중
[OK] OpenAI API 키 설정됨
[ERROR] Anthropic API 키 미설정
[ERROR] Google API 키 미설정
[OK] LangSmith 키 설정됨
LangChain: 1.4.1
LangChain Core: 1.6.3
LangChain OpenAI: 1.6.2


In [4]:
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

documents = [
    Document(
        page_content="면접에서 자기소개를 했다. 컴퓨터 과학을 전공한 신입 개발자이며 자바와 파이썬을 사용했다."
    ),
    Document(
        page_content="프로젝트에서 백엔드 개발자를 맡았다. 사용자 데이터 처리, 서버 로직, RESTful API 구현, 데이터베이스 설계를 담당했다."
    ),
    Document(
        page_content="팀 프로젝트에서 의사소통 문제가 있었다. 정기적인 미팅과 진행 상황 공유를 통해 문제를 해결했다."
    ),
    Document(
        page_content="개발자로서 빠른 학습 능력과 문제 해결 능력이 강점이다. 새로운 기술을 빠르게 습득하고 복잡한 문제에 해결책을 제시한다."
    ),
]

embeddings_model= OpenAIEmbeddings(model = "text-embedding-3-small")

vectorstore= FAISS.from_documents(
    documents,
    embedding=embeddings_model
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 10}
)

question = "백엔드 개발자로 어떤 일을 했나요?"
result = retriever.invoke(question)

for document in result:
    print(document.page_content)


프로젝트에서 백엔드 개발자를 맡았다. 사용자 데이터 처리, 서버 로직, RESTful API 구현, 데이터베이스 설계를 담당했다.
면접에서 자기소개를 했다. 컴퓨터 과학을 전공한 신입 개발자이며 자바와 파이썬을 사용했다.
개발자로서 빠른 학습 능력과 문제 해결 능력이 강점이다. 새로운 기술을 빠르게 습득하고 복잡한 문제에 해결책을 제시한다.
팀 프로젝트에서 의사소통 문제가 있었다. 정기적인 미팅과 진행 상황 공유를 통해 문제를 해결했다.


In [ ]:
from operator import itemgetter

from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model = "gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful chatbot."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

memory = InMemoryChatMessageHistory()

chain = prompt | llm

respnse = chain.invoke({
    "chat_histroy": [],
    "input": "안녕하세요. 제 이름은 테디입니다."
})

print(respnse.content)